## Topic 2: Comparison — FAISS vs. Chroma vs. Qdrant vs. Pinecone vs. Weaviate

### Concept, Intuition, Why It Exists

- Not all "vector search" tools are the same kind of thing — some are libraries, some are embedded databases, some are fully managed cloud services. Knowing the distinction matters because the right choice depends on what stage a project is in, what infrastructure already exists, and what operational requirements actually exist.
- **FAISS** (Facebook AI Similarity Search): a C++ library with a Python wrapper, not a database at all. It provides highly optimized index structures (Flat, IVF, HNSW, PQ) for nearest-neighbor search over vectors stored in memory or on disk. No server, no persistence layer, no metadata filtering, no REST API — just raw index-and-search over numpy arrays. The fastest option at pure search speed for a given index type, but everything else (serialising the index, storing original text, handling metadata, managing persistence across restarts) is your responsibility entirely.
- **Chroma**: an embedded vector database (runs in-process, no separate server needed) designed specifically for rapid prototyping in the LangChain/LlamaIndex ecosystem. Extremely easy to get started — a few lines of Python, no Docker, no cloud account. The trade-off is that it is embedded-first, meaning running it at scale across multiple processes or machines requires stepping outside its simplest use mode. Good for notebooks and small projects, less suitable as a production-grade persistent store with concurrent writers.
- **Qdrant**: an open-source, dedicated vector database written in Rust. Runs as a standalone server (Docker locally, or Qdrant Cloud's managed tier), or in-process via `QdrantClient(":memory:")` with no server needed. Supports HNSW indexing, rich payload (metadata) filtering during search, both REST and gRPC APIs, on-disk storage, collections, and point-level operations. Genuinely production-ready without needing a paid tier — the local Docker image is the same codebase as the cloud offering. This project's choice.
- **Pinecone**: a fully managed, cloud-only vector database — no self-hosting option at all. Simplest operational overhead of any option here (no infrastructure to manage) and scales transparently. Trade-offs: vendor lock-in (your vectors live on Pinecone's servers with no local option), cost at scale (paid tiers), and no way to export stored vectors in a portable format — migrating away means re-embedding the entire corpus from scratch.
- **Weaviate**: an open-source vector database with a broader scope than Qdrant — it has its own built-in vectorisation (can call an embedding model on your behalf at ingest time), a graph-style object schema, and a GraphQL query API alongside REST. More feature-rich than Qdrant but also heavier to configure and operate — the additional features come with real setup complexity overhead.

### Internal Working

- **FAISS**: you manage everything manually — build an index object, add numpy arrays of vectors to it, call `index.search(query_vector, k)` to get back distances and integer IDs, then look up the original texts yourself by mapping those IDs back to your own storage. Serialisation and deserialisation to disk is your responsibility.
- **Chroma**: wraps the storage and retrieval loop — you pass text and metadata at add time (Chroma calls the embedding model you configure), then call `collection.query(query_texts=[...], n_results=k)` and get back documents, metadata, and distances in one call. Embedding, storage, and ID management are handled internally.
- **Qdrant**: you embed text yourself (Qdrant does not call embedding models on your behalf by default), then upsert points (id + vector + payload) into a named collection via REST or gRPC or the in-memory client. At query time, you embed the query yourself and call `client.query_points()` with the query vector, optional payload filter, and k. Qdrant handles HNSW indexing, persistence, and filtering internally.
- **Pinecone**: identical interaction model to Qdrant from the client's perspective — you embed yourself, upsert vectors with metadata, query with an embedded vector and receive top-k matches — but everything runs on Pinecone's managed infrastructure rather than a server you run.
- **Weaviate**: defines a schema upfront (class definitions with typed properties), then adds objects that can be auto-vectorised if a vectoriser module is configured. Queries use either a GraphQL interface or a REST endpoint with `nearVector` or `nearText` operators.

### How It's Implemented In This Project

- This project's earlier in-memory approach was closest to what FAISS provides in its simplest form — raw vectors in a list, brute-force search loop, no persistence.
- Chroma would have been the easiest step up from that (embedded, minimal setup), but was deliberately not chosen — the project's requirements (real metadata filtering, Docker-or-cloud flexibility, a path to production without swapping tools) made Qdrant the better fit, even at the cost of a slightly more involved setup.
- FAISS is not used directly here, but understanding it is important because many other tools (including Qdrant's own index) are built on similar principles to FAISS's HNSW implementation.

### Real-World Issues, Edge Cases, Debugging

- **FAISS's biggest practical pain point is the ID management problem**: FAISS only knows about integer IDs, not text or metadata — you are responsible for maintaining your own mapping from FAISS integer IDs back to the original content. In practice this means a separate dict or database table, and keeping that mapping in sync with the FAISS index is a real operational burden as documents are updated or deleted.
- **Chroma's embedded mode hits limits when multiple processes need to write simultaneously**: since it runs in-process, two separate ingestion workers pointing at the same Chroma store can corrupt it. Chroma does have a client/server mode that solves this, but that is no longer simpler to operate than just running Qdrant directly.
- **Pinecone's vendor lock-in is real and asymmetric**: migrating into Pinecone is easy (their client library handles it). Migrating out means re-embedding your entire corpus from scratch into a new system since Pinecone does not export raw stored vectors in a portable format — worth knowing before committing a large corpus to it.
- **Weaviate's auto-vectorisation is convenient but creates a hidden dependency**: if Weaviate is configured to call an external embedding API on your behalf, your ingest pipeline now has a network call to an external service baked in at ingest time rather than at your own controlled embedding step. A rate limit or outage in that external service blocks ingestion, not a separate visible embedding step you can retry independently.

### Design Decisions & Trade-offs

- FAISS — choose when maximum raw search performance matters and you are willing to own all surrounding infrastructure (persistence, metadata, ID mapping) yourself. Common in research and performance-critical systems with existing infrastructure.
- Chroma — choose when getting something working quickly in a notebook or demo matters more than production readiness. A genuinely good choice for early prototyping; honest about not being the right long-term tool for high-scale or multi-process setups.
- Qdrant — choose when you need production-grade persistence, real metadata filtering, and local-or-cloud flexibility without vendor lock-in and without the overhead of Weaviate's schema and GraphQL layer. This project's choice.
- Pinecone — choose when zero infrastructure management matters more than cost or vendor independence. A reasonable choice for small-to-medium corpora where the managed tier pricing is acceptable.
- Weaviate — choose when you want built-in auto-vectorisation, a richer object schema, and are willing to invest in the configuration complexity that comes with it. More common in enterprise settings with dedicated ML infrastructure teams.

### Alternatives & When To Use Each

- pgvector (Postgres extension) — team already runs Postgres and wants to avoid a new infrastructure component; accepts slightly lower performance than a dedicated vector DB; good when the data is already in Postgres and SQL-level access control matters.
- Redis Vector Search — team already runs Redis and wants to add semantic search without a new system; lower operational overhead than a dedicated vector DB if Redis is already in the stack.
- OpenSearch / Elasticsearch k-NN — team already runs a search stack and wants to add vector search alongside existing full-text search; useful for hybrid keyword + semantic search without a separate vector store.

### Common Mistakes & Production Failures

- Choosing Chroma for production purely because it was the simplest to use in the initial prototype, without verifying it handles the actual production requirements (concurrent writes, large corpus, persistence guarantees).
- Choosing Pinecone without explicitly pricing the corpus size at scale — the cost curve is non-linear and can become significant at hundreds of millions of vectors.
- Using FAISS without a robust ID-to-content mapping layer and discovering after a re-index that the integer IDs no longer map to the same content they did before, because FAISS assigns IDs sequentially and does not handle deletion gracefully.
- Treating all these tools as interchangeable at the code level — switching from Chroma to Qdrant in an existing project is not just a config change, it involves adopting a different data model, different client libraries, and different query semantics.

### Lead-Level Interview Questions

**Q: Your team is three engineers, the corpus is 50,000 chunks, and the project is three months old. Someone proposes Pinecone. What is your actual evaluation?**
A: At 50,000 chunks Pinecone's free or starter tier likely covers it, so cost is not immediately the issue. The real questions are: does this team want to own zero infrastructure, or is running a Docker container acceptable? How likely is a corpus migration to a different system in the next year — if likely, Pinecone's lack of vector export is a meaningful lock-in risk. Is metadata filtering needed during search? There is no universally wrong answer, but the trade-offs should be stated explicitly rather than defaulting to Pinecone because it is the easiest to start.

**Q: FAISS is faster than Qdrant at raw search. When would you still choose Qdrant?**
A: FAISS's speed advantage only holds for pure vector search on vectors that fit in memory, with no metadata filtering, no persistence across restarts, and no multi-client access. The moment you need any of those — and most production systems need at least one — the gap between FAISS's raw search speed and Qdrant's slightly slower but persistent, filterable, multi-client-safe search narrows to irrelevance compared to the operational complexity FAISS leaves entirely to you.

**Q: A teammate says "Weaviate does vectorisation for us, which saves an entire step in the pipeline." Is that a benefit or a risk?**
A: Both. The benefit is real — the ingest pipeline is simpler if you do not manage an embedding step yourself. The risk is also real — you have created an implicit dependency on Weaviate's configured vectoriser module being available, correctly configured, and using the exact same model version at ingest time and query time. Any deviation becomes a silent quality regression or an outage at ingest time. Owning your embedding step explicitly is less convenient but far more debuggable.

### Hidden Concepts Worth Knowing

- **Embedded vs. client-server architecture**: Chroma (embedded mode) runs inside your Python process — simple but means every process that needs the index must load it into its own memory. Qdrant, Pinecone, and Weaviate are all client-server — the index lives in a separate process or service, and multiple clients can query it concurrently without each needing a full copy of the index in memory. This distinction matters as soon as more than one thing needs to read or write the index at the same time.
- **gRPC vs. REST**: Qdrant supports both. REST is easier to debug (human-readable, curl-testable). gRPC is faster (binary protocol, lower serialisation overhead) and worth adopting for high-throughput ingest pipelines. Starting with REST and switching to gRPC when throughput actually matters is the sensible progression.

### Revision Summary

> FAISS is a fast search library, not a database — fast but requires you to own persistence, ID mapping, and metadata entirely. Chroma is an embedded DB, great for prototyping, less suited for production concurrent workloads. Qdrant is open-source, production-ready, locally runnable via Docker or in-process via `:memory:`, and this project's choice. Pinecone is fully managed with zero infrastructure overhead but real vendor lock-in and no vector export. Weaviate adds auto-vectorisation and a richer schema at the cost of more configuration complexity. Choose based on actual requirements — persistence, filtering, scale, infrastructure budget — not on which is most sophisticated.